<img src="https://weclouddata.s3.amazonaws.com/images/logos/wcd_logo_new_2.png"  width='15%'>  


<h1> Demo: LLM Indexing - Getting Started with FAISS</h1>
Developed by WeCloudData
<br></br>


# Importing required libraries

In [ ]:
!nvidia-smi #to review that notebook has GPU access

Tue May 12 11:46:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install faiss-gpu-cu12

print("\n Installation complete.")
print("!!! IMPORTANT: You MUST restart the session now. Go to Runtime > Restart session. !!!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 MB 16.6 MB/s eta 0:00:00

 Installation complete.
!!! IMPORTANT: You MUST restart the session now. Go to Runtime > Restart session. !!!


In [ ]:
!pip install sentence_transformers

## Build Document Vectors

In [ ]:
import requests
from io import StringIO
import pandas as pd

In [ ]:
res = requests.get('https://raw.githubusercontent.com/brmson/dataset-sts/master/data/sts/sick2014/SICK_train.txt')
# create dataframe
data_sick = pd.read_csv(StringIO(res.text), sep='\t')
data_sick.head()

,pair_ID,sentence_A,sentence_B,relatedness_score,entailment_judgment
0,1,A group of kids is playing in a yard and an ol...,A group of boys in a yard is playing and a man...,4.5,NEUTRAL
1,2,A group of children is playing in the house an...,A group of kids is playing in a yard and an ol...,3.2,NEUTRAL
2,3,The young boys are playing outdoors and the ma...,The kids are playing outdoors near a man with ...,4.7,ENTAILMENT
3,5,The kids are playing outdoors near a man with ...,A group of kids is playing in a yard and an ol...,3.4,NEUTRAL
4,9,The young boys are playing outdoors and the ma...,A group of kids is playing in a yard and an ol...,3.7,NEUTRAL


In [ ]:
# we take all samples from both sentence A and B
sentences = data_sick['sentence_A'].tolist()
sentences[:5]

['A group of kids is playing in a yard and an old man is standing in the background',
 'A group of children is playing in the house and there is no man standing in the background',
 'The young boys are playing outdoors and the man is smiling nearby',
 'The kids are playing outdoors near a man with a smile',
 'The young boys are playing outdoors and the man is smiling nearby']

In [ ]:
# we take all samples from both sentence A and B
sentences = data_sick['sentence_A'].tolist()
sentence_b = data_sick['sentence_B'].tolist()
sentences.extend(sentence_b)  # merge them
len(set(sentences))  # together we have ~4.5K unique sentences

4802

In [ ]:
urls = [
    'https://raw.githubusercontent.com/brmson/dataset-sts/master/data/sts/semeval-sts/2012/MSRpar.train.tsv',
    'https://raw.githubusercontent.com/brmson/dataset-sts/master/data/sts/semeval-sts/2012/MSRpar.test.tsv',
    'https://raw.githubusercontent.com/brmson/dataset-sts/master/data/sts/semeval-sts/2012/OnWN.test.tsv',
    'https://raw.githubusercontent.com/brmson/dataset-sts/master/data/sts/semeval-sts/2013/OnWN.test.tsv',
    'https://raw.githubusercontent.com/brmson/dataset-sts/master/data/sts/semeval-sts/2014/OnWN.test.tsv',
    'https://raw.githubusercontent.com/brmson/dataset-sts/master/data/sts/semeval-sts/2014/images.test.tsv',
    'https://raw.githubusercontent.com/brmson/dataset-sts/master/data/sts/semeval-sts/2015/images.test.tsv'
]

In [ ]:
# each of these dataset have the same structure, so we loop through each creating our sentences data
for url in urls:
    res = requests.get(url)
    # extract to dataframe
    data = pd.read_csv(StringIO(res.text), sep='\t', header=None, on_bad_lines='skip')
    # add to columns 1 and 2 to sentences list
    sentences.extend(data[1].tolist())
    sentences.extend(data[2].tolist())

In [ ]:
len(set(sentences))

14505

In [ ]:
# remove duplicates and NaN
sentences = [word for word in list(set(sentences)) if type(word) is str]

In [ ]:
from sentence_transformers import SentenceTransformer
# initialize sentence transformer model
model = SentenceTransformer('all-mpnet-base-v2')
# create sentence embeddings
sentence_embeddings = model.encode(sentences)
sentence_embeddings.shape

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

(14504, 768)

## Faiss

### 1. IndexFlatL2

In [ ]:
import faiss

[ ]
import faiss

[ ]
d = sentence_embeddings.shape[1]

[ ]
index = faiss.IndexFlatL2(d)

[ ]
index.is_trained
True

[ ]
index.add(sentence_embeddings)

[ ]


[]

In [ ]:
d = sentence_embeddings.shape[1]

In [ ]:
index = faiss.IndexFlatL2(d)

In [ ]:
index.is_trained

True

In [ ]:
index.add(sentence_embeddings)

In [ ]:
index.ntotal

14504

In [ ]:
k = 4
xq = model.encode(["Someone sprints with a football"])

In [ ]:
%%time
D, I = index.search(xq, k)  # search
print(I)

[[ 8105  9611 10590  8282]]
CPU times: user 6.1 ms, sys: 792 µs, total: 6.89 ms
Wall time: 9.13 ms


In [ ]:
[sentences[i] for i in I[0]]

['A man in a football uniform is running with a football during a game.',
 'A player is running with the ball',
 'A person playing football is running past an official carrying a football',
 'A football player is running past an official carrying a football']

If we’d rather extract the numerical vectors from Faiss, we can do that too.

In [ ]:
# we have 4 vectors to return (k) - so we initialize a zero array to hold them
import numpy as np
vecs = np.zeros((k, d))
# then iterate through each ID from I and add the reconstructed vector to our zero-array
for i, val in enumerate(I[0].tolist()):
    vecs[i, :] = index.reconstruct(val)

In [ ]:
vecs

array([[-1.59906428e-02, -2.25989409e-02,  2.23864708e-03, ...,
        -2.39133812e-03,  1.75316557e-02,  3.47983725e-02],
       [-2.64145508e-02, -5.86983003e-02, -1.13434847e-02, ...,
        -4.68761101e-03,  2.81645358e-02,  4.27266844e-02],
       [-2.05919165e-02, -2.07966771e-02, -2.31960267e-02, ...,
         1.00538181e-02,  3.41035724e-02,  2.54181270e-02],
       [-6.71151653e-03, -1.78009216e-02, -2.18290333e-02, ...,
         4.21926015e-05,  2.00489685e-02,  2.25975364e-02]])

### 2. FLV

In [ ]:
import faiss

In [ ]:
d = sentence_embeddings.shape[1]

In [ ]:
nlist = 50  # how many cells
quantizer = faiss.IndexFlatL2(d)
index = faiss.IndexIVFFlat(quantizer, d, nlist)

Here we’ve added a new parameter nlist. We use nlist to specify how many partitions (Voronoi cells) we’d like our index to have.

In [ ]:
index.is_trained

False

Now that we added clustering with IndexIVFFlat, we will need to train the index. So, what we do now is train our index on our data — which we must do before adding any data to the index.



In [ ]:
index.train(sentence_embeddings)
index.is_trained  # check if index is now trained

True

Now that our index is trained, we add our data just as we did before.


In [ ]:
index.add(sentence_embeddings)
index.ntotal  # number of embeddings indexed

14504

In [ ]:
k = 4
xq = model.encode(["Someone sprints with a football"])

In [ ]:
%%time
D, I = index.search(xq, k)  # search
print(I)

[[ 8105  9611 10590  8282]]
CPU times: user 1.01 ms, sys: 0 ns, total: 1.01 ms
Wall time: 1.05 ms


The search time has clearly decreased, in this case, we don’t find any difference between results returned by our exhaustive search, and this approximate search. But, often this can be the case.



If approximate search with IndexIVFFlat returns suboptimal results, we can improve accuracy by increasing the search scope. We do this by increasing the nprobe attribute value — which defines how many nearby cells to search.

>

In [ ]:
index.nprobe = 10

In [ ]:
%%time
D, I = index.search(xq, k)  # search
print(I)

[[ 8105  9611 10590  8282]]
CPU times: user 2.07 ms, sys: 0 ns, total: 2.07 ms
Wall time: 3.27 ms


### 3. Product Quantization


1.   Split the original vector into several subvectors.
2.   For each set of subvectors, perform a clustering operation — creating multiple centroids for each sub-vector set.
3.   In the vector of sub-vectors, replace each sub-vector with the ID of it’s nearest set-specific centroid.






In [ ]:
import faiss

In [ ]:
m = 8  # number of centroid IDs in final compressed vectors
bits = 8 # number of bits in each centroid

quantizer = faiss.IndexFlatL2(d)  # we keep the same L2 distance flat index
index = faiss.IndexIVFPQ(quantizer, d, nlist, m, bits)

In [ ]:
index.is_trained



False

In [ ]:
index.train(sentence_embeddings)



In [ ]:
index.add(sentence_embeddings)



In [ ]:
index.nprobe = 10  # align to previous IndexIVFFlat nprobe value



In [ ]:
%%time
D, I = index.search(xq, k)
print(I)

[[9611  140 7539 3388]]
CPU times: user 1.37 ms, sys: 0 ns, total: 1.37 ms
Wall time: 1.96 ms


In [ ]:
[sentences[i] for i in I[0]]

['A player is running with the ball',
 'A sports uniform is being worn by a man who is running down the field',
 'A man is wearing a sports uniform and is running down the field',
 'A man who is playing is running with the ball in his hands']